# H&M Mini Dataset Exploration

This notebook loads the three tables of `data/H_and_M_mini_dataset` (`articles`, `customers`, `transactions`), first with **PyArrow** and then with **pandas**, and shows the head of each DataFrame.

# 1. Setup

Imports, logging and the dataset location shared by both loading approaches.

This cell imports the libraries, configures logging (the notebook is an entry point, so configuring it here is fine) and resolves the dataset directory relative to the repo root, so the notebook works no matter where Jupyter was started.

In [2]:
import logging
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger("h_and_m_exploration")

# Walk up from the working directory until the folder containing data/ is found
REPO_ROOT: Path = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "H_and_M_mini_dataset").exists()
)
DATASET_DIR: Path = REPO_ROOT / "data" / "H_and_M_mini_dataset"
TABLE_NAMES: list[str] = ["articles", "customers", "transactions"]

logger.info("dataset_dir=%s tables=%d", DATASET_DIR, len(TABLE_NAMES))

INFO dataset_dir=f:\git-projects\ecommerce-customer-support-agents\data\H_and_M_mini_dataset tables=3


# 2. Load with PyArrow

Each table is a folder of Parquet files. `transactions` is additionally Hive-partitioned (`year=.../month=.../`), and PyArrow turns those folder names into `year` and `month` columns automatically.

This cell defines a helper that reads one table folder into a `pyarrow.Table` and uses it to load all three tables. PyArrow ignores the `_SUCCESS` marker files, so pointing at the folder is enough.

In [3]:
def load_table_pyarrow(dataset_dir: Path, table_name: str) -> pa.Table:
    """Read one table folder of Parquet files into a PyArrow table.

    Args:
        dataset_dir (Path): Root folder of the H&M mini dataset.
        table_name (str): Sub-folder name of the table (e.g. "articles").

    Returns:
        pa.Table: All rows of the table, including Hive partition columns if any.

    Raises:
        FileNotFoundError: If the table folder does not exist.
    """
    table_path = dataset_dir / table_name
    if not table_path.exists():
        raise FileNotFoundError(f"Missing table folder: {table_path}")

    # Reading the folder makes PyArrow discover every Parquet file and partition
    table = pq.read_table(table_path)
    logger.info("loader=pyarrow table=%s rows=%d cols=%d", table_name, table.num_rows, table.num_columns)
    return table


arrow_tables: dict[str, pa.Table] = {name: load_table_pyarrow(DATASET_DIR, name) for name in TABLE_NAMES}

INFO loader=pyarrow table=articles rows=10 cols=25
INFO loader=pyarrow table=customers rows=5 cols=7
INFO loader=pyarrow table=transactions rows=10 cols=7


# 3. Load with pandas

Now the same three tables as pandas DataFrames, and the head of each.

This cell loads the tables into pandas. `pd.read_parquet` accepts a folder and uses PyArrow as its engine, so the result matches the tables loaded above.

In [4]:
def load_table_pandas(dataset_dir: Path, table_name: str) -> pd.DataFrame:
    """Read one table folder of Parquet files into a pandas DataFrame.

    Args:
        dataset_dir (Path): Root folder of the H&M mini dataset.
        table_name (str): Sub-folder name of the table (e.g. "customers").

    Returns:
        pd.DataFrame: All rows of the table, including Hive partition columns if any.
    """
    df = pd.read_parquet(dataset_dir / table_name)
    logger.info("loader=pandas table=%s rows=%d cols=%d", table_name, len(df), df.shape[1])
    return df


dataframes: dict[str, pd.DataFrame] = {name: load_table_pandas(DATASET_DIR, name) for name in TABLE_NAMES}

INFO loader=pandas table=articles rows=10 cols=25
INFO loader=pandas table=customers rows=5 cols=7
INFO loader=pandas table=transactions rows=10 cols=7


## 3.1 Head of each DataFrame

A quick look at the first rows of `articles`, `customers` and `transactions`.

This cell prints the table name and displays the first rows of each DataFrame so the columns and value formats can be checked.

In [5]:
for name, df in dataframes.items():
    print(f"=== {name} {df.shape} ===")
    display(df.head())

=== articles (10, 25) ===


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,0351484002,0351484,Lazer Razer Brief,59,Swimwear bottom,Swimwear,1010016,Solid,09,Black,...,Swimwear,B,Lingeries/Tights,1,Ladieswear,60,"Womens Swimwear, beachwear",1018,Swimwear,"Fully lined bikini bottoms with a mid waist, m..."
1,0399061032,0399061,Jacket Slim,262,Jacket,Garment Upper body,1010023,Denim,09,Black,...,Outwear,A,Ladieswear,1,Ladieswear,19,Womens Jackets,1007,Outdoor,"Fitted jacket in washed, stretch denim with bu..."
2,0469682004,0469682,Dolores (1),245,Cardigan,Garment Upper body,1010016,Solid,09,Black,...,Knitwear Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1003,Knitwear,Cardigan in a soft rib knit with a shawl colla...
3,0507883014,0507883,Ellen Thong (Malva) L 3pk,286,Underwear bottom,Underwear,1010010,Melange,08,Dark Grey,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear",NaN
4,0542464001,0542464,Buckle Up Triangle Top,298,Bikini top,Swimwear,1010016,Solid,09,Black,...,Swimwear,B,Lingeries/Tights,1,Ladieswear,60,"Womens Swimwear, beachwear",1018,Swimwear,Triangle bikini top with decorative straps at ...


=== customers (5, 7) ===


,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,0cc38999a55a5057620301060dd75bb9f89094bfe1e0f2...,1.0,1.0,ACTIVE,Regularly,53.0,f7096ad22304216a7257c7774f84234061c3c8e03265b5...
1,3d3f5c0c29bd0102405a4f069d1c38c05ccafe92c99e6e...,NaN,NaN,ACTIVE,NONE,49.0,f576bb3c368dbcdbfd7dad578818fafe335892e9ec6a73...
2,b95a44a38e584b79ce5c5d7fbec35483fd7ebb424d5df0...,1.0,1.0,ACTIVE,Regularly,27.0,240fc1cc9ed2bea936b4085c56af477ab7124d4cf812e4...
3,d653802971d076d838ce1df3b1b6885653bace224b9052...,1.0,1.0,ACTIVE,Regularly,53.0,6ae185fdb852d8c9983f3d0accffc20c922c89b8bccbc7...
4,f43c9ccc4ff2fe25dda1758b5c91b967045ec8c13ffb4a...,1.0,1.0,ACTIVE,Regularly,49.0,d991c8eb6f0da0100cd029af383258c62b9060fc2fb0f8...


=== transactions (10, 7) ===


,customer_id,t_dat,article_id,price,sales_channel_id,year,month
0,d653802971d076d838ce1df3b1b6885653bace224b9052...,2018-10-21,0469682004,0.033881,2,2018,10
1,d653802971d076d838ce1df3b1b6885653bace224b9052...,2019-09-08,0805275001,0.067780,2,2019,9
2,d653802971d076d838ce1df3b1b6885653bace224b9052...,2019-09-11,0570189003,0.025407,2,2019,9
3,d653802971d076d838ce1df3b1b6885653bace224b9052...,2020-03-17,0728156006,0.050831,2,2020,3
4,f43c9ccc4ff2fe25dda1758b5c91b967045ec8c13ffb4a...,2020-03-08,0507883014,0.008458,2,2020,3


# 4. Save as CSV

Each table is written to `data/H_and_M_mini_dataset/csv-format/<table>.csv`, so the data can be opened in tools that do not read Parquet.

This cell defines a helper that writes one DataFrame to CSV (creating the output folder if needed) and saves all three tables. The index is skipped because it carries no information.

In [6]:
CSV_DIR: Path = DATASET_DIR / "csv-format"


def save_table_csv(df: pd.DataFrame, output_dir: Path, table_name: str) -> Path:
    """Write one DataFrame to `<output_dir>/<table_name>.csv`.

    Args:
        df (pd.DataFrame): Table to save.
        output_dir (Path): Folder to write into; created if it does not exist.
        table_name (str): Name of the table, used as the CSV file name.

    Returns:
        Path: Location of the written CSV file.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    csv_path = output_dir / f"{table_name}.csv"

    df.to_csv(csv_path, index=False)
    logger.info("saver=csv table=%s rows=%d path=%s", table_name, len(df), csv_path)
    return csv_path


csv_paths: dict[str, Path] = {name: save_table_csv(df, CSV_DIR, name) for name, df in dataframes.items()}

INFO saver=csv table=articles rows=10 path=f:\git-projects\ecommerce-customer-support-agents\data\H_and_M_mini_dataset\csv-format\articles.csv
INFO saver=csv table=customers rows=5 path=f:\git-projects\ecommerce-customer-support-agents\data\H_and_M_mini_dataset\csv-format\customers.csv
INFO saver=csv table=transactions rows=10 path=f:\git-projects\ecommerce-customer-support-agents\data\H_and_M_mini_dataset\csv-format\transactions.csv
